In [1]:
import pandas as pd
import numpy as np 
import torch 

In [2]:
import sys

In [3]:
package_path = '/home/gridsan/tmackey/cdvae/scripts/1-22-2024_clean_impelementations/'

# Add the package path to sys.path
if package_path not in sys.path:
    sys.path.insert(0, package_path)

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from eval_utils import load_data

from collections import Counter
import os
import numpy as np
from tqdm import tqdm

from pymatgen.core.structure import Structure
from pymatgen.core.composition import Composition
from pymatgen.core.lattice import Lattice

from pymatgen.analysis.diffraction.xrd import XRDCalculator
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
from pymatgen.io.cif import CifWriter

from pymatgen.analysis.structure_analyzer import SpacegroupAnalyzer

from compute_metrics import get_crystal_array_list, all_results_retreival, Crystal

from eval_utils import (
    smact_validity, structure_validity, CompScaler, get_fp_pdist,
    load_config, load_data, get_crystals_list, compute_cov)

def load_and_merge(data_dir, label, total_data = None, num_workers = 100):
    for worker in range(num_workers): 
        filename = label + '_worker' + str(worker) + '.npy'
        if worker == 0: 
            total_data = np.load(data_dir + '/' + filename)
        else:
            total_data_chunk = np.load(data_dir + '/' + filename)
            #stack data
            total_data = np.vstack((total_data, total_data_chunk))
    
    return total_data

#replaced args.label with label
def get_file_paths(root_path, task, label='', suffix='pt'):
    if label == '':
        out_name = f'eval_{task}.{suffix}'
    else:
        out_name = f'eval_{task}_{label}.{suffix}'
    out_name = os.path.join(root_path, out_name)
    return out_name

def list_cif_writer(data_dir, predicted_crystal_list, gt_structure_list, symprec = None):
    for i in tqdm(range(len(predicted_crystal_list))): 
        crystal = Crystal(predicted_crystal_list[i])    
        structure = crystal.structure
        full_formula = structure.composition.formula
        file_path = data_dir + '/pred_{}_{}.cif'.format(full_formula, i)
        if symprec is not None: 
            CifWriter(structure, symprec = symprec).write_file(file_path)
        CifWriter(structure).write_file(file_path)
    
    for i in tqdm(range(len(gt_structure_list))):
        structure = Crystal(gt_structure_list[i]).structure
        full_formula = structure.composition.formula
        file_path = data_dir + '/gt_{}_{}.cif'.format(full_formula, i)
        if symprec is not None: 
            CifWriter(structure, symprec = symprec).write_file(file_path)
        CifWriter(structure).write_file(file_path)

def MAPE(y_true, y_pred): 
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def cosine_similarity(patternx, patterny):
    return np.dot(patternx, patterny) / (np.linalg.norm(patternx) * np.linalg.norm(patterny))

def mape_calculator(gt_crystals, best_crystals): 
    #calculate the MAPE for the lengths and angles
    per_crystal_mape_lengths = []   
    per_crystal_mape_angles = []
    for i in range(len(best_crystals)): 
        gt_lengths = gt_crystals[i]['lengths']
        gt_angles = gt_crystals[i]['angles']
        pred_lengths = best_crystals[i]['lengths']
        pred_angles = best_crystals[i]['angles']
        per_crystal_mape_lengths.append(MAPE(gt_lengths, pred_lengths))
        per_crystal_mape_angles.append(MAPE(gt_angles, pred_angles))

    return per_crystal_mape_lengths, per_crystal_mape_angles

def hist_plot(data, title, xlabel, ylabel):
    plt.hist(data, bins = 100)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

def rmsd_and_mards_plotter(lowest_rmsd, per_crystal_mape_lengths, per_crystal_mape_angles):
    #plot a histogram of the rmsds 
    hist_plot(lowest_rmsd, 'RMSD for Crystal Structures', 'RMSD', 'Count')
    
    #plot a histogram of the MAPEs for the lengths
    hist_plot(per_crystal_mape_lengths, 'MAPE for Crystal Structure Lengths', 'MAPE', 'Count')

    #plot a histogram of the MAPEs for the angles
    hist_plot(per_crystal_mape_angles, 'MAPE for Crystal Structure Angles', 'MAPE', 'Count')

def get_best_crystals(total_rmsd, all_gt, all_results, plot = True, use_lp_metrics = ""):
    
    nan_rows = np.argwhere(np.isnan(total_rmsd))
    rows_with_only_nans = np.where(np.all(np.isnan(total_rmsd), axis=1))[0]
    
    list_of_all_lattice_mapes = []
    #for every batch 
    for i in range(len(all_results)): 
        #fetch the gt and pred crystals for that batch
        gt_crystals = all_gt[i]
        pred_crystals = all_results[i]

        #calculate the MAPE for the lengths and angles
        per_crystal_mape_lengths, per_crystal_mape_angles = mape_calculator(gt_crystals, pred_crystals)
        list_of_all_lattice_mapes.append(per_crystal_mape_lengths)
        
    #apply np array
    list_of_all_lattice_mapes = np.array(list_of_all_lattice_mapes)
    #reshape by transposing
    list_of_all_lattice_mapes = list_of_all_lattice_mapes.T

    #if a value is nan in total rmsd, set it to a very large number in the list of all lattice mapes
    list_of_all_lattice_mapes[nan_rows[:, 0], nan_rows[:, 1]] = 1000000

    if use_lp_metrics != "":
        #take the nan min to get the lowest metric and lowest metric indices
        lowest_metric = np.nanmin(list_of_all_lattice_mapes, axis=1)
        lowest_metric_indices = np.nanargmin(np.nan_to_num(list_of_all_lattice_mapes, nan=1), axis=1)

    else:
        lowest_metric = np.nanmin(total_rmsd, axis=1)
        lowest_metric_indices = np.nanargmin(np.nan_to_num(total_rmsd, nan=1), axis=1)
        
    #use these values to find the best crystal from all_results
    best_crystals = []
    for i in range(len(all_results[0])): 
        indice = lowest_metric_indices[i]
        best_crystals.append(all_results[indice][i])

    gt_crystals = all_gt[0]

    #remove the values with only nans (no matches)
    best_crystals = [i for j, i in enumerate(best_crystals) if j not in rows_with_only_nans]
    gt_crystals = [i for j, i in enumerate(gt_crystals) if j not in rows_with_only_nans]

    #use the lowest metric indices to fetch the corresponding rmsd values 
    lowest_rmsd = []
    for i in range(len(lowest_metric_indices)): 
        lowest_rmsd.append(total_rmsd[i][lowest_metric_indices[i]])
    lowest_rmsd = np.array(lowest_rmsd)

    #use the lowest metric indices to fetch the corresponding mapes
    lowest_mapes = []
    for i in range(len(lowest_metric_indices)): 
        lowest_mapes.append(list_of_all_lattice_mapes[i][lowest_metric_indices[i]])
    lowest_mapes = np.array(lowest_mapes)

    return lowest_metric, best_crystals, gt_crystals, lowest_rmsd, lowest_mapes

def symmetryops(structure, symprec):
    sga = SpacegroupAnalyzer(structure, symprec=symprec)
    space_group_symbol = sga.get_space_group_symbol()
    space_group_number = sga.get_space_group_number()  # Get the space group number
    symmetrized_structure = sga.get_refined_structure()
    return space_group_symbol, space_group_number, symmetrized_structure

def symmetry_performance(snapped_crystal_list, pred_crystal_list, gt_crystal_list, tolmax = 2):
    snapped_spacegroups = []
    pred_spacegroups = []
    snapped_tolerance = []
    pred_tolerance = []

    for i in tqdm(range(len(snapped_crystal_list))): 
        snapped_found = False
        pred_found = False

        snapped_structure = Crystal(snapped_crystal_list[i]).structure
        if pred_crystal_list is not None:
            pred_structure = Crystal(pred_crystal_list[i]).structure
        gt_structure = Crystal(gt_crystal_list[i]).structure
        
        gt_spacegroup, gt_sg_num, gt_symmetrized_structure = symmetryops(gt_structure, 0.01)

        for symprec in np.arange(0.01, tolmax, 0.1):
            try: 
                if not snapped_found: 
                    snapped_spacegroup, snapped_sg_num, snapped_symmetrized_structure = symmetryops(snapped_structure, symprec)
                if not pred_found and pred_crystal_list is not None:
                    pred_spacegroup, pred_sg_num, pred_symmetrized_structure = symmetryops(pred_structure, symprec)
            except: 
                continue

            if snapped_sg_num == gt_sg_num and not snapped_found:
                snapped_tolerance.append(symprec)
                snapped_spacegroups.append(snapped_sg_num)
                snapped_found = True

            if pred_crystal_list is not None: 
                if pred_sg_num == gt_sg_num and not pred_found:
                    pred_tolerance.append(symprec)
                    pred_spacegroups.append(pred_sg_num)
                    pred_found = True

            if snapped_found and pred_found:
                break            

            if symprec == np.max(np.arange(0.01, tolmax, 0.1)):
                if not snapped_found: 
                    snapped_spacegroups.append(snapped_sg_num)
                    snapped_tolerance.append(-1)

                if not pred_found and pred_crystal_list is not None:
                    pred_spacegroups.append(pred_sg_num)
                    pred_tolerance.append(-1)
                    
    snapped_spacegroups = np.array(snapped_spacegroups)
    snapped_tolerance = np.array(snapped_tolerance)

    if pred_crystal_list is not None:
        pred_spacegroups = np.array(pred_spacegroups)
        pred_tolerance = np.array(pred_tolerance)

    return snapped_spacegroups, pred_spacegroups, snapped_tolerance, pred_tolerance

In [ ]:
# config_dict = {
#     "model_path": '/home/gridsan/tmackey/hydra/singlerun/2024-01-29/augmented_vae_nopf',
#     "label": 'PDF_unsolved_compounds',
#     "num_batches": 64,
#     "data_dir": '/home/gridsan/tmackey/cdvae/scripts/1-22-2024_clean_impelementations/Result_Data/Fig3', 
#     "results_label": 'fs_total_rmsd_ae_pf_5batches_64_evals',
# }

# config_dict = {
#     "model_path": '/home/gridsan/tmackey/hydra/singlerun/2024-01-29/vae_nopf',
#     "label": '5_batches_64_evals',
#     "num_batches": 64,
#     "data_dir": '/home/gridsan/tmackey/cdvae/scripts/1-22-2024_clean_impelementations/Result_Data/Fig3', 
#     "results_label": 'fs_total_rmsd_ae_pf_5batches_64_evals',
# }

config_dict = {
    "model_path": '/home/gridsan/tmackey/hydra/singlerun/2024-01-29/augmented_vae_nopf',
    "label": 'RRUFF_data_test_only_using_amcsd',
    "num_batches": 64,
    "data_dir": '/home/gridsan/tmackey/cdvae/scripts/1-22-2024_clean_impelementations/Result_Data/Fig3', 
    "results_label": 'RRUFF_data_test_only_using_amcsd',
}

In [ ]:
model_path = config_dict['model_path']
label = config_dict['label']
recon_file_path = get_file_paths(model_path, 'recon',label=label)
num_batches = config_dict['num_batches']

all_results, all_gt = all_results_retreival(recon_file_path, False, num_batches, label = label)

In [ ]:
result_label = "RRUFF"
for i in range(config_dict['num_batches']):
    folder_name = f"/home/gridsan/tmackey/cdvae/scripts/1-22-2024_clean_impelementations/02-10-2024_RRUFF_snap/test_{i}_structures_{result_label}"
    #make the directory 
    os.makedirs(folder_name, exist_ok=True)
    predicted_structures = all_results[i]
    gt_structures = all_gt[i]
    list_cif_writer(folder_name, predicted_structures, gt_structures, symprec = 0.001)